In [ ]:
RESIZE = 356
CROP = 299

In [ ]:
def save_checkpoint(state, filename="/kaggle/input/best_model/pytorch/default/1/best_model (1).pt"):
    print("=> Saving checkpoint")
    torch.save(state, filename)


def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    step = checkpoint["step"]
    return step

In [ ]:
import os
import pandas as pd
import spacy  # for tokenizer
import torch
from torch.nn.utils.rnn import pad_sequence  # pad batch
from torch.utils.data import DataLoader, Dataset , random_split
from PIL import Image  # Load img
from tqdm import tqdm
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import sys
import numpy as np
import torch.nn.functional as F
import re
import numpy as np
import matplotlib.pyplot as plt
import torchmetrics as tm
import torch.nn as nn

from collections import Counter

torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
def num_trainable_params(model):
  nums = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
  return nums

# **DATASET**

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/flickr-image-dataset/flickr30k_images/results.csv',
    sep='|',
    header=None,
    skiprows=1,
    names=['ID', 'idx', 'caption']
)

# Add image path
img_path = '/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images/'
df['image'] = img_path + df['ID']

# Group captions by image ID
grouped = df.groupby(['ID', 'image'])['caption'].apply(list).reset_index()

df = grouped

df.head(5)


In [ ]:
# Check for missing values in the caption column
print(df['caption'].isnull().sum())

# Display rows with missing captions
print(df[df['caption'].isnull()])
df = df.dropna(subset=['caption']).reset_index(drop=True)
df['caption'] = df['caption'].astype(str)
df.head(5)

In [ ]:
df['caption'] = df['caption'].str.lower()
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
grouped_data = df.groupby('image')['caption'].apply(lambda group: group.values.tolist()).reset_index(name='captions')

df_train , df_test = train_test_split(df,test_size=0.2,shuffle=True)
df_test , df_valid = train_test_split(df_test,test_size=0.5,shuffle=True)

df_train.head(5)

# **VOCAB**

In [ ]:
# Load spaCy tokenizer
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "tagger"])

# Tokenizer function
def tokenizer(text):
    return [token.text.lower() for token in nlp(text) if not token.is_space]


In [ ]:
captions = df_train['caption'].tolist()

In [ ]:
# Special tokens
specials = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']

# Count token frequencies
counter = Counter()
for caption in captions:
    tokens = tokenizer(caption)
    counter.update(tokens)

# Sort tokens (optional: by frequency or alphabetically)
sorted_tokens = sorted(counter.items(), key=lambda x: -x[1])

# Create vocab: word → index
vocab = {tok: idx + len(specials) for idx, (tok, _) in enumerate(sorted_tokens)}
for i, sp in enumerate(specials):
    vocab[sp] = i

# Inverse mapping: index → word
inv_vocab = {idx: tok for tok, idx in vocab.items()}

In [ ]:
print("Vocabulary size:", len(vocab))
print("Sample tokens:", list(vocab.items())[:50])


In [ ]:
unk_index = vocab['<UNK>']

def numericalize(tokens):
    return [vocab.get(tok, unk_index) for tok in tokens]

In [ ]:
word = "<EOS>"
index = vocab.get(word, vocab['<UNK>'])  # returns <UNK> index if word not in vocab
print(f"Index of '{word}':", index)


In [ ]:
sample = "This is an unknown wordlikeobject."
tokens = tokenizer(sample)
ids = numericalize(tokens)

print("Tokens:", tokens)
print("Token IDs:", ids)


In [ ]:
import pickle

# Save
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# # Load
# with open("vocab.pkl", "rb") as f:
#     vocab = pickle.load(f)

# **FlickrDataset_MYCOLLATE**

In [ ]:
class FlickrDataset(Dataset):
    def __init__(self, root_dir, captions_file,vocab, transform=None):
        self.root_dir = root_dir
        self.df = captions_file
        self.tokenizer = tokenizer
        self.vocab = vocab
        self.transform = transform

        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        self.sos = self.vocab["<SOS>"]
        self.eos = self.vocab["<EOS>"]
        self.unk = self.vocab["<UNK>"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions.iloc[index]
        img_id = self.imgs.iloc[index]
        

        try:
            img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_id}: {e}")
            img = Image.new("RGB", (224, 224))

        if self.transform:
            img = self.transform(img)

        tokens = self.tokenizer(caption)
        numericalized_caption = [self.sos] + [self.vocab.get(tok, self.unk) for tok in tokens] + [self.eos]

        return img, torch.tensor(numericalized_caption)


In [ ]:
class MyCollate:
    def __init__(self, pad_idx, max_len=100):
        self.pad_idx = pad_idx
        self.max_len = max_len  # Optional maximum caption length

    def __call__(self, batch):
        # Stack images: each is [3, H, W]
        images = [img.unsqueeze(0) for img, _ in batch]
        images = torch.cat(images, dim=0)  # [B, 3, H, W]

        # Optionally truncate captions to max_len
        captions = []
        for _, caption in batch:
            if self.max_len is not None:
                caption = caption[:self.max_len]  # Truncate
            captions.append(caption)

        # Pad to length of the longest caption in the batch (or max_len if all were truncated)
        captions_padded = pad_sequence(captions, batch_first=True, padding_value=self.pad_idx)
        # Shape: [max_len, batch_size]

        return images, captions_padded

In [ ]:
def get_loader(root_folder,annotation_file,vocab,transform,batch_size=32,num_workers=8,shuffle=True,pin_memory=True):
    
    dataset = FlickrDataset(
        root_dir=root_folder,
        captions_file=annotation_file,
        vocab=vocab,
        transform=transform
    )

    pad_idx = vocab["<PAD>"]

    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=shuffle,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )

    return loader, dataset


In [ ]:
batch_size = 32
transform = transforms.Compose(
    [
        transforms.Resize((RESIZE, RESIZE)),
        transforms.RandomCrop((CROP, CROP)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
    ]
)
train_loader, dataset = get_loader(
    root_folder = f'/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images/',
    annotation_file = df_train,
    vocab = vocab,
    transform=transform,
    num_workers=4,
    batch_size=batch_size,
    shuffle=True,
)

valid_loader, dataset = get_loader(
    root_folder = f'/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images/',
    annotation_file = df_valid,
    vocab = vocab,
    transform=transform,
    num_workers=4,
    batch_size=batch_size,
    shuffle=True,
)



# **Embedding**

In [ ]:
from gensim.models import Word2Vec
embedding_dim =  300
# Assuming you trained it already
model = Word2Vec(captions, vector_size=embedding_dim, window=20, min_count=5, workers=4)
word_vectors = model.wv

In [ ]:
embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, idx in vocab.items():
    if word in word_vectors:
        embedding_matrix[idx] = word_vectors[word]
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)


# **MODEL**

In [ ]:
import torchvision.models as models
import math

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad = False  # Freeze ResNet weights
        modules = list(resnet.children())[:-1]  # Remove final FC
        self.resnet = nn.Sequential(*modules)
        self.fc = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        features = self.resnet(images)  # [B, 2048, 1, 1]
        features = features.view(features.size(0), -1)  # [B, 2048]
        features = self.fc(features)  # [B, E]
        features = self.bn(features)
        return features  # [B, E]


class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2) * -(math.log(10000.0) / embed_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)  # [max_len, 1, E]
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0)]


class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads, num_layers,
                 embedding_weights=None, dropout=0.1, max_len=100):
        super(DecoderTransformer, self).__init__()

        if embedding_weights is not None:
            self.embed = nn.Embedding.from_pretrained(embedding_weights, freeze=False, padding_idx=0)
            embed_size = embedding_weights.size(1)  # Update embed size if needed
        else:
            self.embed = nn.Embedding(vocab_size, embed_size)

        self.pos_encoder = PositionalEncoding(embed_size, max_len)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, features, captions):
        # captions: [B, T] → [T, B, E]
        tgt = self.embed(captions).permute(1, 0, 2)
        tgt = self.pos_encoder(tgt)

        memory = features.unsqueeze(0)  # [1, B, E]
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(0)).to(tgt.device)

        out = self.transformer_decoder(tgt, memory, tgt_mask=tgt_mask)
        out = self.fc_out(out)  # [T, B, V]
        return out.permute(1, 0, 2)  # [B, T, V]




class ImageCaptioningModel(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads=8, num_layers=3,
                 dropout=0.1, embedding_weights=None):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = EncoderCNN(embed_size)
        self.decoder = DecoderTransformer(
            embed_size=embed_size,
            vocab_size=vocab_size,
            num_heads=num_heads,
            num_layers=num_layers,
            dropout=dropout,
            embedding_weights=embedding_weights
        )

    def forward(self, images, captions):
        features = self.encoder(images)
        return self.decoder(features, captions)



In [ ]:
vocab_size = len(vocab)  # Size of your vocabulary
num_heads = 8            # Number of attention heads
num_layers = 3           # Number of Transformer decoder layers
dropout = 0.1            # Dropout rate


model = ImageCaptioningModel(
    embed_size=embedding_tensor.size(1),
    vocab_size=len(vocab),
    num_heads=10,
    num_layers=3,
    dropout=0.1,
    embedding_weights=embedding_tensor
).to(device)


In [ ]:
x_batch, y_batch = next(iter(train_loader))
x_batch.shape, y_batch.shape

In [ ]:
y_batch[1]

In [ ]:
metric = tm.text.Perplexity().to(device)

In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, metric,epoch=None, clip=0.5):
    model.train()
    loss_train = AverageMeter()
    metric.reset()

    with tqdm.tqdm(train_loader, unit='batch') as tepoch:
        for images, captions in tepoch:
            if epoch is not None:
                tepoch.set_description(f"Epoch {epoch}")

            images = images.to(device)
            captions = captions.to(device)

            # Prepare inputs and targets for teacher forcing
            captions_input = captions[:, :-1]      # remove <EOS>
            captions_target = captions[:, 1:]      # remove <SOS>

            # Forward pass
            outputs = model(images, captions_input)  # [B, T, V]

            # Compute loss
            loss = loss_fn(
                outputs.reshape(-1, outputs.size(-1)),     # [B*T, V]
                captions_target.reshape(-1)             # [B*T]
            )

            # Backpropagation
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            loss_train.update(loss.item(), n=captions.size(0))
            metric.update(outputs.detach(), captions_target)  # [B, T, V] and [B, T]

            tepoch.set_postfix(loss=loss_train.avg, metric=metric.compute().item())

    return model, loss_train.avg, metric.compute().item()


In [ ]:
def evaluate(model, test_loader, loss_fn, metric, pad_idx=None):
    model.eval()
    loss_eval = AverageMeter()
    metric.reset()

    with torch.no_grad():
        for images, captions in test_loader:
            images = images.to(device)
            captions = captions.to(device)

            # Prepare inputs and targets
            captions_input = captions[:, :-1]
            captions_target = captions[:, 1:]

            # Forward pass
            outputs = model(images, captions_input)  # [B, T, V]

            # Compute loss
            loss = loss_fn(
                outputs.reshape(-1, outputs.size(-1)),         # [B*T, V]
                captions_target.reshape(-1)                 # [B*T]
            )

            # Logging
            loss_eval.update(loss.item(), n=captions.size(0))
            metric.update(outputs.detach(), captions_target)

    return loss_eval.avg, metric.compute().item()


In [ ]:
from torch.optim import AdamW , SGD
lr = 0.005
wd = 1e-4
optimizer = SGD(model.parameters(), lr= lr, weight_decay=wd , momentum = 0.9)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"])

In [ ]:
loss_train_hist = []
loss_valid_hist = []

metric_train_hist = []
metric_valid_hist = []

best_loss_valid = torch.inf
epoch_counter = 0

In [ ]:
num_epochs = 2
def train(model ,train_loader , val_loader, loss_fn , optimizer):

    global loss_train_hist, loss_valid_hist, metric_train_hist, metric_valid_hist
    global epoch_counter, best_loss_valid 

    for epoch in range(num_epochs):
      # Train
      model, loss_train, metric_train = train_one_epoch(model,
                                                     train_loader,
                                                     loss_fn,
                                                     optimizer,
                                                     metric,
                                                     epoch)
      # Validation
      loss_valid, metric_valid = evaluate(model,
                                         valid_loader,
                                         loss_fn,
                                         metric)
    
      loss_train_hist.append(loss_train)
      loss_valid_hist.append(loss_valid)
    
      metric_train_hist.append(metric_train)
      metric_valid_hist.append(metric_valid)
    
      if loss_valid < best_loss_valid:
        torch.save(model, f'model.pt')
        best_loss_valid = loss_valid
        print('Model Saved!')
    
      print(f'Valid: Loss = {loss_valid:.4}, Metric = {metric_valid:.4}')
      print()
    
      epoch_counter += 1

In [ ]:
import tqdm
clip = 0.25

In [ ]:
train(model ,train_loader , valid_loader, loss_fn , optimizer)

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(range(epoch_counter), loss_train_hist, 'r-', label='Train')
plt.plot(range(epoch_counter), loss_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('loss')
plt.grid(True)
plt.legend()

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(range(epoch_counter), metric_train_hist, 'r-', label='Train')
plt.plot(range(epoch_counter), metric_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('loss')
plt.grid(True)
plt.legend()

In [ ]:
test_loader, test_dataset = get_loader(
    root_folder = f'/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images/',
    annotation_file = df_test,
    vocab = vocab,
    transform=transform,
    num_workers=4,
    batch_size=32,
    shuffle=True,
)    
model.eval()


In [ ]:
import torch.nn.functional as F

def test(model, test_loader, loss_fn, metric, pad_idx=None, temperature=1.0):
    model.eval()
    loss_test = AverageMeter()
    metric.reset()

    all_outputs = []
    all_targets = []
    all_images = []

    with torch.no_grad():
        for images, captions in test_loader:
            images = images.to(device)
            captions = captions.to(device)

            captions_input = captions[:, :-1]
            captions_target = captions[:, 1:]

            outputs = model(images, captions_input)  # [B, T, V]

            # Apply temperature before decoding
            if temperature != 1.0:
                outputs = outputs / temperature

            # Greedy decoding with temperature-scaled logits
            predictions = outputs.argmax(dim=-1).cpu()

            # Compute loss & metric
            loss = loss_fn(outputs.reshape(-1, outputs.size(-1)),
                           captions_target.reshape(-1))
            loss_test.update(loss.item(), n=captions.size(0))
            metric.update(outputs.detach(), captions_target)

            all_outputs.append(predictions)
            all_targets.append(captions_target.cpu())
            all_images.append(images.cpu())

    return loss_test.avg, metric.compute().item(), all_outputs, all_targets, all_images


In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T

def decode_caption(tokens, vocab):
    inv_vocab = {idx: word for word, idx in vocab.items()}
    words = []
    for token in tokens:
        word = inv_vocab.get(token.item(), "<UNK>")
        if word == "<EOS>":
            break
        if word not in ["<PAD>", "<SOS>"]:
            words.append(word)
    return " ".join(words)



In [ ]:
def show_predictions(images, preds, targets, vocab, num_examples=5,
                     mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):

    unnormalize = T.Normalize(mean=[-m / s for m, s in zip(mean, std)],
                              std=[1 / s for s in std])

    for i in range(min(num_examples, len(images))):
        img_tensor = images[i]

        # Handle nested tensors like [1, 3, H, W]
        if isinstance(img_tensor, torch.Tensor):
            while img_tensor.dim() > 3:
                img_tensor = img_tensor.squeeze(0)
        else:
            continue  # Skip if not a tensor

        try:
            img = unnormalize(img_tensor)
            img = img.permute(1, 2, 0).cpu().numpy().clip(0, 1)
        except Exception as e:
            print(f"Error processing image {i}: {e}")
            continue

        pred_text = decode_caption(preds[i], vocab)
        target_text = decode_caption(targets[i], vocab)

        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Pred: {pred_text}\nTrue: {target_text}", fontsize=10)
        plt.show()


In [ ]:
test_loss, test_metric, predictions, targets, images = test(
    model, test_loader, loss_fn, metric, temperature=0.5
)

show_predictions(
    images=images[0],         # list or tensor of images [B, 3, H, W]
    preds=predictions[0],     # list of predicted token IDs [B, T]
    targets=targets[0],       # list of ground truth token IDs [B, T]
    vocab=vocab,           # dictionary: token → ID
    num_examples=5         # show 5 image-caption pairs
)



In [ ]:
import torch.nn.functional as F

def generate_caption(model, image, vocab, max_len=20, temperature=1.0, sos_token="<SOS>", eos_token="<EOS>"):
    model.eval()
    
    inv_vocab = {idx: word for word, idx in vocab.items()}
    sos_idx = vocab[sos_token]
    eos_idx = vocab[eos_token]
    
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)  # [1, 3, H, W]
        features = model.encoder(image)        # [1, embed_size]

        generated = [sos_idx]
        for _ in range(max_len):
            input_seq = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)  # [1, T]
            logits = model.decoder(features, input_seq)  # [1, T, V]
            next_token_logits = logits[:, -1, :] / temperature  # [1, V]
            next_token = torch.argmax(next_token_logits, dim=-1).item()

            if next_token == eos_idx:
                break
            generated.append(next_token)

        caption = [inv_vocab.get(idx, "<UNK>") for idx in generated[1:]]  # remove <SOS>
        return " ".join(caption)


In [ ]:
def show_tensor_image_with_caption(img_tensor, caption,
                                   mean=(0.485, 0.456, 0.406),
                                   std=(0.229, 0.224, 0.225)):

    # Unnormalize
    unnormalize = T.Normalize(
        mean=[-m/s for m, s in zip(mean, std)],
        std=[1/s for s in std]
    )
    img = unnormalize(img_tensor.squeeze(0) if img_tensor.dim() == 4 else img_tensor)
    img = img.permute(1, 2, 0).cpu().numpy().clip(0, 1)

    # Display
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    print(caption)


In [ ]:
for i in range(4):
    # Get image tensor (already transformed)
    img_tensor, _ = test_dataset[300+i]

    # Generate caption
    caption = generate_caption(model, img_tensor, vocab, temperature=0.5)

    # Show image with caption
    show_tensor_image_with_caption(img_tensor, caption)


In [ ]:
from PIL import Image

# Load and preprocess the image
image_pil1 = Image.open("/kaggle/input/test-photo/henley-3.jpg").convert("RGB")
image_pil2 = Image.open("/kaggle/input/test-photo/istockphoto-892646738-612x612.jpg").convert("RGB")
def load_image(image):
    transform = T.Compose([
        T.Resize((224, 224)),  # match training
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    
    img_tensor = transform(image)
    
    # Generate and show caption
    caption = generate_caption(model, img_tensor, vocab, temperature=0.5)
    show_tensor_image_with_caption(img_tensor, caption)
    
load_image(image_pil1)
load_image(image_pil2)